### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Q-Learning is the most important classical RL algorithm — it's the foundation that DQN (RL/03), Actor-Critic (RL/04), and PPO (RL/05) all build upon. Once you understand Q-values and the TD update rule, everything else is an extension.

**Mental Model:** Imagine learning to drive in a new city. You don't have a map (no model). You try different routes (exploration), and over time you learn which turns are good (high Q-value) and which lead to dead ends (low Q-value). Eventually, you know the best route from any intersection to your destination — that's Q-learning.

**Common Junior Pitfall:** Setting exploration rate (ε) too low too fast. The agent finds a "pretty good" policy early and sticks with it forever, missing the OPTIMAL policy. Always decay ε slowly and never set it to exactly 0.

---

## 📑 Table of Contents

  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1 · Monte Carlo vs Temporal Difference](#1-monte-carlo-vs-temporal-difference)
  * [Two Ways to Learn Without a Model](#two-ways-to-learn-without-a-model)
* [2 · Q-Values: Rating Actions, Not Just States](#2-q-values-rating-actions-not-just-states)
* [3 · Q-Learning & SARSA Algorithm](#3-q-learning-sarsa-algorithm)
  * [Q-Learning Update Rule (Off-Policy)](#q-learning-update-rule-off-policy)
  * [SARSA Update Rule (On-Policy)](#sarsa-update-rule-on-policy)
* [4 · Exploration vs Exploitation](#4-exploration-vs-exploitation)
  * [The Fundamental Tradeoff](#the-fundamental-tradeoff)
  * [Common Strategies](#common-strategies)
  * [Upper Confidence Bound (UCB)](#upper-confidence-bound-ucb)
* [✅ Knowledge Check](#knowledge-check)
  * [Q1: On-policy vs off-policy](#q1-on-policy-vs-off-policy)
  * [Q2: Q-table limitations](#q2-q-table-limitations)
  * [Q3: Epsilon decay](#q3-epsilon-decay)
* [🔨 Practical Practice](#practical-practice)
  * [Exercise 1: Frozen Lake](#exercise-1-frozen-lake)
  * [Exercise 2: Multi-Armed Bandit with UCB](#exercise-2-multi-armed-bandit-with-ucb)
  * [Exercise 3: Taxi Environment](#exercise-3-taxi-environment)


In [ ]:
!pip install -q numpy matplotlib gymnasium pygame

## 1 · Monte Carlo vs Temporal Difference

### Two Ways to Learn Without a Model

**Monte Carlo:** Wait until the episode ENDS, then update values based on the actual return.

$$V(s_t) \leftarrow V(s_t) + \alpha \left[ \underbrace{G_t}_{\text{actual return}} - V(s_t) \right]$$

**Temporal Difference (TD):** Update IMMEDIATELY after each step using an ESTIMATE of the return.

$$V(s_t) \leftarrow V(s_t) + \alpha \left[ \underbrace{r_{t+1} + \gamma V(s_{t+1})}_{\text{estimated return (TD target)}} - V(s_t) \right]$$

| | Monte Carlo | Temporal Difference |
|---|---|---|
| Updates when? | End of episode | Every step |
| Needs episodes to end? | Yes | No |
| Variance | High (full returns are noisy) | Low (single-step estimates) |
| Bias | Unbiased | Biased (uses estimated V) |
| Best for | Short episodes | Continuing tasks |

**Key insight:** The TD update $r + \gamma V(s')$ is called "bootstrapping" — using your own estimate to update yourself. This is the same idea as the Bellman equation, but learned from experience instead of computed from the model.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

# Compare MC vs TD learning using Gymnasium's CliffWalking environment
def td_value_learning(env, episodes=500, alpha=0.1, gamma=0.9):
    """Learn V(s) using Temporal Difference (TD(0)) for a random policy."""
    n_states = env.observation_space.n
    V = np.zeros(n_states)
    history = []
    
    for ep in range(episodes):
        state, info = env.reset()
        total_reward = 0
        
        for _ in range(100):
            action = env.action_space.sample()  # Random policy
            next_state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            total_reward += reward
            
            # TD Update: V(s) += α * [r + γV(s') - V(s)]
            td_target = reward + gamma * V[next_state] * (1 - done)
            td_error = td_target - V[state]
            V[state] = V[state] + alpha * td_error
            
            state = next_state
            if done:
                break
        
        history.append(total_reward)
    
    return V, history

env = gym.make('CliffWalking-v1')
V_td, rewards_td = td_value_learning(env, episodes=1000)

print('TD(0) Learned Value Function (reshaped to grid):')
print(np.round(V_td.reshape(4, 12), 1))
print(f'\nNotice the low values near the bottom row (the cliff)!')

---
## 2 · Q-Values: Rating Actions, Not Just States

V(s) tells you how good a STATE is. But to ACT, you need to know how good each ACTION is in that state.

$$Q(s, a) = \text{"How good is it to take action } a \text{ in state } s \text{?"}$$

The relationship:
$$V(s) = \max_a Q(s, a)$$

If you know Q-values, the optimal policy is trivial:
$$\pi^*(s) = \arg\max_a Q(s, a) \quad \text{(just pick the action with highest Q)}$$

This is why Q-learning is so powerful — learn the Q-table, and the policy falls out for free.

---
## 3 · Q-Learning & SARSA Algorithm

### Q-Learning Update Rule (Off-Policy)

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ \underbrace{r + \gamma \max_{a'} Q(s', a')}_{\text{TD target}} - Q(s, a) \right]$$

Q-learning uses $\max_{a'} Q(s', a')$ — the BEST possible action from $s'$. This assumes the agent will act optimally from the next state onwards, regardless of what it ACTUALLY does (which includes random exploration). This makes Q-learning **off-policy**.

### SARSA Update Rule (On-Policy)
SARSA (State-Action-Reward-State-Action) updates using the action the agent EXACTLY takes next:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ \underbrace{r + \gamma Q(s', a')}_{\text{TD target}} - Q(s, a) \right]$$

This makes SARSA **on-policy**. It learns the value of the policy it is CURRENTLY following, including its exploratory randomness.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

class RLAgent:
    def __init__(self, n_states, n_actions, alpha=0.1, gamma=0.99, epsilon=1.0):
        self.Q = np.zeros((n_states, n_actions))
        self.n_actions = n_actions
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
    
    def choose_action(self, state):
        if np.random.random() < self.epsilon:
            return np.random.randint(self.n_actions)
        return np.argmax(self.Q[state])

class QLearningAgent(RLAgent):
    def learn(self, state, action, reward, next_state, done):
        q_current = self.Q[state][action]
        q_next_max = 0 if done else np.max(self.Q[next_state])
        td_target = reward + self.gamma * q_next_max
        self.Q[state][action] += self.alpha * (td_target - q_current)

class SARSAAgent(RLAgent):
    def learn(self, state, action, reward, next_state, next_action, done):
        q_current = self.Q[state][action]
        q_next = 0 if done else self.Q[next_state][next_action]
        td_target = reward + self.gamma * q_next
        self.Q[state][action] += self.alpha * (td_target - q_current)

def train_agent(agent_type, episodes=500):
    env = gym.make('CliffWalking-v1')
    n_states = env.observation_space.n
    n_actions = env.action_space.n
    agent = agent_type(n_states, n_actions, epsilon=0.1) # constant e for clear comparison
    
    rewards = []
    for _ in range(episodes):
        state, _ = env.reset()
        action = agent.choose_action(state)
        total_reward = 0
        
        while True:
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            next_action = agent.choose_action(next_state)
            
            if isinstance(agent, QLearningAgent):
                agent.learn(state, action, reward, next_state, done)
            else:
                agent.learn(state, action, reward, next_state, next_action, done)
                
            total_reward += reward
            state, action = next_state, next_action
            if done:
                break
        rewards.append(total_reward)
    return agent, rewards

# Train both agents
q_agent, q_rewards = train_agent(QLearningAgent, 500)
sarsa_agent, sarsa_rewards = train_agent(SARSAAgent, 500)

# Learning Curves Plot
window = 20
smoothed_q = [np.mean(q_rewards[max(0,i-window):i+1]) for i in range(len(q_rewards))]
smoothed_sarsa = [np.mean(sarsa_rewards[max(0,i-window):i+1]) for i in range(len(sarsa_rewards))]

fig, axes = plt.subplots(1, 1, figsize=(10, 5))
axes.plot(smoothed_q, label='Q-Learning (Off-policy)', color='coral')
axes.plot(smoothed_sarsa, label='SARSA (On-policy)', color='steelblue')
axes.set_xlabel('Episode')
axes.set_ylabel('Smoothed Total Reward')
axes.set_title('SARSA vs Q-Learning on CliffWalking-v1')
axes.legend()
axes.grid(True)
plt.tight_layout()
plt.savefig('sarsa_vs_q_learning.png', dpi=100)
plt.show()

# Print Learned Policies Visualization
def print_policy(agent, name):
    arrows = {0: '↑', 1: '→', 2: '↓', 3: '←'}
    print(f'\n{name} Learned Policy (epsilon=0 for evaluation):')
    for i in range(4):
        row = ''
        for j in range(12):
            if i == 3 and 0 < j < 11:
                row += ' X '
            elif i == 3 and j == 11:
                row += ' ★ '
            else:
                state = i * 12 + j
                best = arrows[np.argmax(agent.Q[state])]
                row += f' {best} '
        print(f'  {row}')
    if isinstance(agent, QLearningAgent):
        print("Notice: Q-learning takes the risky path along the cliff! (assumes it won't make mistakes).")
    else:
        print("Notice: SARSA takes the safe path far from the cliff! (accounts for random exploration).")

print_policy(q_agent, 'Q-Learning')
print_policy(sarsa_agent, 'SARSA')

---
## 4 · Exploration vs Exploitation

### The Fundamental Tradeoff

- **Exploit:** Take the best known action → good short-term reward
- **Explore:** Try random actions → might find something better

If you always exploit, you might miss the optimal path. If you always explore, you never use what you've learned.

### Common Strategies

| Strategy | How It Works | Pros | Cons |
|----------|-------------|------|------|
| **ε-greedy** | Random action with prob ε | Simple, effective | Explores uniformly (even bad actions) |
| **ε-decay** | ε decreases over time | Explores early, exploits late | Need to tune decay rate |
| **Boltzmann** | Sample actions by softmax(Q/τ) | Explores good actions more | Temperature tuning |
| **UCB** | Bonus for unexplored actions | Theoretically optimal | More complex |

### Upper Confidence Bound (UCB)
Instead of exploring randomly uniformly, UCB pulls actions that have high uncertainty (fewer pull counts). 

$$A_t = \arg\max_a \left[ Q(a) + c \sqrt{\frac{\ln t}{N(a)}} \right]$$

In [ ]:
import numpy as np

class UCBAgent:
    def __init__(self, n_actions, c=2.0):
        self.n_actions = n_actions
        self.c = c # Exploration constant
        self.Q = np.zeros(n_actions)
        self.N = np.zeros(n_actions) # Action counts
        self.t = 0
    
    def choose_action(self):
        self.t += 1
        # If any action has count 0, select it to establish a baseline
        if 0 in self.N:
            return np.argmin(self.N)
        
        # Calculate UCB score
        ucb_values = self.Q + self.c * np.sqrt(np.log(self.t) / self.N)
        return np.argmax(ucb_values)
    
    def update(self, action, reward):
        self.N[action] += 1
        # Simple incremental average equation
        self.Q[action] += (reward - self.Q[action]) / self.N[action]

print("UCB snippet implemented! Contrast this with the random selection in epsilon-greedy agent.")
print("UCB ensures mathematically that optimal actions are discovered without excessive exploration of known sub-optimal paths.")

---

## ✅ Knowledge Check

### Q1: On-policy vs off-policy
SARSA uses the actual next action $a'$ in its update. How does this differ from Q-learning?

<details><summary>👉 Answer</summary>

SARSA is ON-policy: it updates toward the value of the action it ACTUALLY took (including exploratory random actions). Q-learning is OFF-policy: it updates toward the BEST possible action (max). SARSA learns a safer policy (accounts for its own randomness), Q-learning learns the optimal policy (assumes perfect execution). SARSA is better for real-time control where you can't afford risky exploration.
</details>

### Q2: Q-table limitations
Why can't you use a Q-table for a game with 10 million possible states?

<details><summary>👉 Answer</summary>

The table would have 10M rows × n_actions columns — too large to store and too slow to fill (you'd need to visit each (state, action) pair many times). Solution: replace the table with a NEURAL NETWORK that approximates Q(s,a). Input: state, output: Q-values for all actions. This is DQN (RL/03). The network generalizes across similar states.
</details>

### Q3: Epsilon decay
What goes wrong if you never decay epsilon (always ε=0.5)?

<details><summary>👉 Answer</summary>

The agent takes random actions 50% of the time FOREVER, even after learning the optimal policy. Evaluation performance would be poor because half the actions are random. You need to decay ε so the agent gradually shifts from exploration to exploitation. Common schedule: ε = max(0.01, ε × 0.995).
</details>

---

## 🔨 Practical Practice

### Exercise 1: Frozen Lake
Use Gymnasium's `FrozenLake-v1`. Test both Q-learning and SARSA in a slippery environment. Does the risk-aversion of SARSA perform better on ice compared to Q-learning?

### Exercise 2: Multi-Armed Bandit with UCB
Expand the UCB snippet above to work against a 10-armed bandit environment where each arm returns rewards from a different normal distribution. Plot its performance vs an epsilon-greedy bandit.

### Exercise 3: Taxi Environment
Implement Q-learning for Gymnasium's `Taxi-v3` environment (500 states, 6 actions). Track how many episodes are needed to achieve optimal performance without dropping the passenger.

---

**Next →** RL 03: Deep Q-Networks (DQN)